# Project Scope



## Recap:
My selected dataset is CIC-IDS-2018, which is a benchmark network intrusion detection dataset produced by the Canadian Institute for Cybersecurity. The specific file capture used was from Friday, February 3rd containing 1,048,575 network records across 80 features. Features are low-level network statistics like packet counts, byte volumes, and inter-arrival times (IAT) that are extracted from raw PCAP captures. This file contains Benign (762,384) and Bot (286,191) traffic classes.

## Key Findings:
I noticed a large class imbalance where there is majority benign traffic and bot traffic is the minority. This could imply unsupervised methods may be less biased than supervised classifiers. Certain features were also heavily skewed to the right which needed clipping to be better visualized. I also noted feature redundancy in the correlation between IAT types and packet lengths. This suggests the 80 features create significant redundancy and that I can compact that down into a non-redundant feature set. I also found that aggregating flows into 1 minute windows revealed traffic patterns that showed bursts. This suggests that ordering of flows within time windows carries important information.

## Course Techniques
For course techniques, I can use Frequent Itemset Mining and Clustering. For Frequent Itemset Mining, I'll bin numeric flow features into categories like Low, Medium, and High, then can use Apriori to find which combinations of these bins frequently appear together in bot vs benign flows. For clustering, I'll use Kmeans and DBSCAN to see if the flows naturally group in a way that separates bot from benign traffic without using labels at all.

## External Techniques
An external technique I can use is Sequential Pattern Mining (PrefixSpan). This differs from Apriori because PrefixSpan mines for ordered sequences within time windows whereas Apriori treats each flow as an unordered set. Bot traffic is defined by repeated connection sequences over time so ordering matters in a way that itemset mining alone cannot capture.

# Research Question Definition

## **RQ1:** What combinations of discretized flow features frequently co-occur in bot traffic compared to benign traffic, and can these combinations serve as distinguishing signatures?
- Data mining task type: Frequent itemset mining, association rule mining
- Relevant algorithm: Apriori
- Evaluation Criteria: Support, confidence, lift
## **RQ2:** Do Kmeans and DBSCAN naturally separate bot from benign without using labels, and do the resulting clusters align with the true class distribution?
- Data mining task type: Clustering
- Relevant algorithm: Kmeans, DBSCAN
- Evaluation Criteria: Adjusted Rand Index (ARI) against true labels, silhouette score, visual cluster purity
## **RQ3:** Do sequential flow patterns within 1 minute time windows reveal temporal signatures of bot traffic that unordered itemset mining misses?
- Data mining task type: Sequential pattern mining
- Relevant algorithm: PrefixSpan
- Evaluation Criteria: Sequential support, pattern length, contrast with RQ1 itemset results to measure what ordering adds

# Motivation and Feasibility

## **Motivation:**
Initial EDA showed 3 properties of this dataset that make these RQs worth asking. First, the bot and benign flows have different feature distributions. This suggests that co-occurrence patterns and cluster structure should differ between classes. Second, the feature redundancy across IAT and packet length variants means the 80-feature space can be compacted without losing discriminative information. Lastly, aggregating flows into 1 minute windows showed bursts in traffic patterns. This suggests that the ordering of flows within time windows delivers important information.
## **Non-Triviality:**
Looking just at individual feature distributions does not tell you which combinations of features matter together, which is what itemset mining addresses. Clustering is non-trivial because it never sees the labels, so alignment with the true class distribution is not guaranteed. Sequential pattern mining goes further than itemset mining by retaining the ordering of flows, which Apriori ignores.
## **Feasibility:**
All three methods can be run on this dataset. Apriori is available through mlxtend, K-Means and DBSCAN through scikit-learn, and PrefixSpan through the prefixspan Python package. The dataset is large but taking a sample that includes both Bot and Benign flows keeps the size manageable.
## **Risks:**
Apriori can be slow on wide datasets so feature pruning from my initial notebook correlation analysis will be applied before running it. Kmeans assumes spherical clusters which may not reflect the true shape of the data, and DBSCAN results change a lot depending on its distance threshold. If bot flows are too spread out across time windows, then sequences may be too short to find meaningful patterns and support threshold tuning will be needed for PrefixSpan.


In [ ]:
%pip install mlxtend -q
%pip install prefixspan -q
%pip install seaborn -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\austi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\austi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\austi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


df = pd.read_csv(
    "/content/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv",
    low_memory=False,

)

assert df.shape[0] > 0 and df.shape[1] > 0, "dataFrame is empty after loading"
assert "Label" in df.columns, "expected 'Label' column not found"

df["Label"] = df["Label"].astype(str).str.strip()
df.loc[df["Label"].isin(["", "nan", "NaN", "None"]), "Label"] = np.nan
missing_labels = df["Label"].isna().sum()
print("missing labels before drop: ", missing_labels)

df = df.dropna(subset=["Label"])

<>:7: DeprecationWarning: invalid escape sequence '\D'
<>:7: DeprecationWarning: invalid escape sequence '\D'
C:\Users\austi\AppData\Local\Temp\ipykernel_14352\218115961.py:7: DeprecationWarning: invalid escape sequence '\D'
  "C:\Dataset\Friday-02-03-2018_TrafficForML_CICFlowMeter.csv",


missing labels before drop:  0


In [ ]:
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import adjusted_rand_score, silhouette_score
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from prefixspan import PrefixSpan

df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df = df.dropna(subset=["Timestamp"]).sort_values("Timestamp").reset_index(drop=True)

num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

print(f"dataset ready: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(df["Label"].value_counts())

dataset ready: 1,048,575 rows, 80 columns
Label
Benign    762384
Bot       286191
Name: count, dtype: int64


In [ ]:
#rq1 feasibility test: frequent itemset mining

TEST_N = 2500
test_df = pd.concat([
    df[df["Label"] == "Benign"].sample(n=TEST_N, random_state=42),
    df[df["Label"] == "Bot"].sample(n=TEST_N, random_state=42)
])

#selected a small number of features to discretize
TEST_FEATURES = ["Flow Pkts/s", "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts"]

disc_test = pd.DataFrame()
for feat in TEST_FEATURES:
    short = feat.replace(" ", "_").replace("/", "_per_")

    #trying 3 bins, if not do 2 bins
    try:
        binned = pd.qcut(
            test_df[feat], q=3,
            labels=["Low", "Med", "High"],
            duplicates="drop"
        )
    except ValueError:
        binned = pd.qcut(
            test_df[feat], q=2,
            labels=["Low", "High"],
            duplicates="drop"
        )
    disc_test[feat] = short + "_" + binned.astype(str)

#convert to transaction format and run apriori
te = TransactionEncoder()
te_array = te.fit_transform(disc_test.values.tolist())
te_df = pd.DataFrame(te_array, columns=te.columns_)

#apriori with minsup = .2
freq_items = apriori(te_df, min_support=0.2, use_colnames=True)
print(f"frequent itemsets found: {len(freq_items)}")
print(freq_items.sort_values("support", ascending=False).head(10))

assert len(freq_items) > 0, "apriori returned no itemsets"
print("apriori test successful")

frequent itemsets found: 38
    support                                         itemsets
7    0.5178                    frozenset({Tot_Bwd_Pkts_Low})
9    0.5026                    frozenset({Tot_Fwd_Pkts_Low})
6    0.4822                   frozenset({Tot_Bwd_Pkts_High})
26   0.4754  frozenset({Tot_Fwd_Pkts_Low, Tot_Bwd_Pkts_Low})
1    0.3338                   frozenset({Flow_Duration_Low})
0    0.3334                  frozenset({Flow_Duration_High})
4    0.3334                 frozenset({Flow_Pkts_per_s_Low})
3    0.3334                frozenset({Flow_Pkts_per_s_High})
5    0.3332                 frozenset({Flow_Pkts_per_s_Med})
2    0.3328                   frozenset({Flow_Duration_Med})
apriori test successful


The test run found 38 frequent itemsets at minsup=.2. The most common single items were low packet counts and low flow duration, which makes sense since most network flows are short. This confirms Apriori runs on this dataset and produces meaningful itemsets worth investigating deeper.

In [ ]:
#rq2 feasibility test: clustering

TEST_FEATURES_CLUSTER = ["Flow Pkts/s", "Flow Duration",
                           "Tot Fwd Pkts", "Tot Bwd Pkts",
                           "Pkt Len Mean", "Flow IAT Mean"]

test_cluster = pd.concat([
    df[df["Label"] == "Benign"].sample(n=TEST_N, random_state=42),
    df[df["Label"] == "Bot"].sample(n=TEST_N, random_state=42)
])

X = test_cluster[TEST_FEATURES_CLUSTER].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

km = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = km.fit_predict(X_scaled)

#comparing cluster assignments against try labels using ARI
true_labels = (test_cluster["Label"] == "Bot").astype(int).values
ari = adjusted_rand_score(true_labels, cluster_labels)
sil = silhouette_score(X_scaled, cluster_labels)

print(f"kmeans ARI: {ari:.4f}")
print(f"silhouette score: {sil:.4f}")
print("cluster counts:", pd.Series(cluster_labels).value_counts().to_dict())

assert -1 <= ari <= 1, "ARI out of expected range"
print("clustering pilot successful")

kmeans ARI: 0.0000
silhouette score: 0.9813
cluster counts: {0: 4999, 1: 1}
clustering pilot successful


The test run produced an ARI of 0.0000 and placed 4,999 out of 5,000 flows into one cluster. Kmeans failed to separate bot from benign with this initial feature set. This is an interesting finding since it suggests that feature selection will be important for RQ2 and that the full analysis may need to try different features or a different value of k.

In [ ]:
#rq3 feasibility test: sequential pattern mining

bot_df = df[df["Label"] == "Bot"].sample(n=5000, random_state=42).copy()
bot_df["window"] = bot_df["Timestamp"].dt.floor("1min")

bot_df["Flow_Rate_Bin"] = pd.qcut(
    bot_df["Flow Pkts/s"], q=3,
    labels=["Low", "Med", "High"],
    duplicates="drop"
).astype(str)

#keeping windows with >= 2 flows
sequences = [
    list(grp["Flow_Rate_Bin"].values)
    for _, grp in bot_df.groupby("window")
    if len(grp) >= 2
]

print(f"number of sequences: {len(sequences)}")

#setting minsup to 50% to speedup runtime
ps = PrefixSpan(sequences)
results = sorted(ps.frequent(len(sequences) // 2), key=lambda x: x[0], reverse=True)

print(f"frequent sequential patterns found: {len(results)}")
for sup, pattern in results[:10]:
    print(f"  support={sup} | {pattern}")

assert len(results) > 0, "prefixspan returned no patterns"
print("prefixspan pilot successful")

number of sequences: 335
frequent sequential patterns found: 59
  support=323 | ['High']
  support=321 | ['Low']
  support=309 | ['Med']
  support=288 | ['Low', 'Low']
  support=285 | ['High', 'Low']
  support=285 | ['Low', 'High']
  support=278 | ['High', 'High']
  support=278 | ['Low', 'Med']
  support=277 | ['Med', 'Low']
  support=274 | ['High', 'Med']
prefixspan pilot successful


This test run built 335 sequences from 1 minute time windows and found 59 frequent sequential patterns. The most common patterns were single items like High and Low flow rates, with two item sequences like Low, Low and High, Low also appearing frequently. This confirms PrefixSpan runs on this dataset and that bot traffic produces repeatable sequential patterns within time windows.

# Methodological Planning

## **Course Algorithms:**
For RQ1, I will use Apriori to mine frequent itemsets from discretized flow features. I chose Apriori because it is straightforward to interpret and works well on the transaction format in the test run. For RQ2, I will use KMeans and DBSCAN. KMeans is a good starting point for finding general cluster structure and DBSCAN complements it by finding dense regions flagging outliers without needing to specify the number of clusters upfront.
## **External Algorithms:**
For RQ3, I will use PrefixSpan to mine sequential patterns from 1 minute time windows. PrefixSpan is the right choice because it is specifically designed for ordered sequences, which Apriori cannot handle. The test run confirmed it runs on this dataset and produces meaningful patterns.
## **Evaluation:**
For RQ1, I will evaluate patterns using support, confidence, and lift. For RQ2, I will use ARI against true labels and silhouette score to measure how well clusters align with the real class distribution. For RQ3, I will evaluate sequential support and pattern length, and compare results against RQ1 to measure what ordering adds beyond unordered itemsets.
## **Baselines:**
For RQ1, the baseline is the top single item frequent itemset. If combinations do not score higher than individual items, itemset mining does not give meaningful information. For RQ2, the baseline is random cluster assignment, which should produce an ARI of 0. The test run already showed this with the initial feature set which leads to making feature selection the next key step. For RQ3, the baseline is the unordered itemset results from RQ1. Sequential patterns are only useful if they reveal something itemset mining missed.

On my honor, I declare the following resources:

1. Collaborators:
- I referenced my earlier homeworks to get started on EDA analysis.
2. AI Tools:
- I used ChatGPT to help with lowering my runtimes for RQ3. I needed to reduce bot sample size and raise minsup threshold. I also used it for grammar/spelling.
3. Resources:
- https://github.com/chuanconggao/PrefixSpan-py

GitHub Link: https://github.com/austinpinedo/ids-anomaly-detection